In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/work/data/vessel_catalog.csv'):
    os.system('mkdir -p /content/work/data /content/work/model /content/work/output /content/raw /content/baseline')
    os.system('unzip -o -q /content/drive/MyDrive/vessel/open_data.zip -d /content/raw')
    os.system('unzip -o -q /content/raw/open.zip -d /content/work/data')
    os.system('unzip -o -q /content/raw/baseline_submit.zip -d /content/baseline')
print(os.listdir('/content/work/data'))

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

D = '/content/work/data'
train = pd.read_csv(f'{D}/train.csv')
cat   = pd.read_csv(f'{D}/vessel_catalog.csv')
sub   = pd.read_csv(f'{D}/sample_submission.csv')

print(train.shape, cat.shape, sub.shape)

In [ ]:
print('== train 컬럼')
print(train.dtypes)
print('\n== catalog 컬럼')
print(cat.dtypes)

display(train.head(10))
display(cat.head(10))

In [ ]:
m = train.merge(cat, left_on='target_vessel_id', right_on='vessel_id', suffixes=('_rec', '_cat'))

cols = ['vessel_name_rec', 'vessel_name_cat',
        'gross_tonnage_rec', 'gross_tonnage_cat',
        'flag_name_rec', 'flag_name_cat',
        'vessel_type_rec', 'vessel_type_cat']
print(m[cols].sample(20, random_state=0).to_string())

In [ ]:
import re

# 한글 선박명 기록 비율
m['is_kor'] = m['vessel_name_rec'].astype(str).str.contains('[가-힣]')
print('한글 선박명 비율:', round(m['is_kor'].mean(), 3))

# 총톤수: 결측 비율, 정확히 일치하는 비율, 상대 차이 분포
gt_rec, gt_cat = m['gross_tonnage_rec'], m['gross_tonnage_cat']
print('기록 톤수 결측 비율:', round(gt_rec.isna().mean(), 3))
both = gt_rec.notna() & gt_cat.notna()
print('톤수 정확히 일치:', round((gt_rec[both] == gt_cat[both]).mean(), 3))
rel = (gt_rec[both] - gt_cat[both]).abs() / gt_cat[both]
print('톤수 상대 차이 분포:')
print(rel.describe(percentiles=[.5, .75, .9, .95, .99]).round(3))

In [ ]:
!ls -laR /content/baseline

In [ ]:
!cat /content/baseline/script.py
!echo '-----'
!cat /content/baseline/requirements.txt